<a href="https://colab.research.google.com/github/nivethithanm/mini-claw/blob/main/CLAW_01_llm_core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLAW-01 — LLM Core: Chat Loop & Streaming

**Goal:** Build the beating heart of your agent — a robust chat loop powered by the OpenAI API.

By the end of this notebook you'll have:
- A `ChatSession` class that manages conversation history
- Streaming responses (like a real terminal agent)
- A system prompt engine — the personality layer
- Token counting and context window management
- A bare-bones REPL you can talk to

> **First principles mindset:** An LLM agent is just a stateful function:  
> `response = f(system_prompt, history, user_message)`  
> Everything else in OpenClaw is built on top of this.


## 0. Setup & Dependencies

In [4]:
!pip install openai tiktoken --quiet

In [3]:
# Install if needed
# !pip install openai tiktoken

import os
import json
import time
from typing import Optional, Generator
from dataclasses import dataclass, field

# Set your key — use env var in production, never hardcode
# os.environ["OPENAI_API_KEY"] = "sk-..."

from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("OpenAI client ready ✓")


OpenAI client ready ✓


## 1. Message Schema

OpenAI uses a simple message list format. Let's wrap it cleanly.

```
[
  {"role": "system",    "content": "You are a helpful assistant."},
  {"role": "user",      "content": "Hello"},
  {"role": "assistant", "content": "Hi! How can I help?"},
  ...
]
```

**Exercise:** Implement `Message` and `Conversation` below.


In [5]:
from dataclasses import dataclass, field
from typing import Literal

Role = Literal["system", "user", "assistant", "tool"]

@dataclass
class Message:
    role: Role
    content: str

    def to_dict(self) -> dict:
        return {"role": self.role, "content": self.content}


@dataclass
class Conversation:
    """
    Holds the full message history for one session.

    Why a class and not just a list?
    - We want token counting
    - We want truncation logic
    - We want serialization for persistence (needed in NB-03)
    """
    system_prompt: str
    messages: list[Message] = field(default_factory=list)

    def add(self, role: Role, content: str):
        # TODO: append a new Message to self.messages
        self.messages.append(Message(role, content))

    def to_api_payload(self) -> list[dict]:
        """
        Returns the full message list in OpenAI API format,
        with the system prompt as the first message.
        """
        # TODO: prepend system message, then return all messages as dicts
        payload = [Message(role='system', content=self.system_prompt).to_dict()] + [m.to_dict() for m in self.messages]
        return payload

    def last_n(self, n: int) -> "Conversation":
        """Return a new Conversation with only the last n messages."""
        # TODO: used for sliding window truncation
        new = Conversation(system_prompt=self.system_prompt)
        new.messages = self.messages[-n:]
        return new

    def __len__(self):
        return len(self.messages)


In [6]:
# Quick smoke test
conv = Conversation(system_prompt="You are a friendly assistant.")
conv.add("user", "Hello!")
conv.add("assistant", "Hi there!")
conv.add("user", "What time is it?")

payload = conv.to_api_payload()
assert payload[0]["role"] == "system"
assert len(payload) == 4   # system + 3 messages
print("Conversation tests pass ✓")


Conversation tests pass ✓


## 2. Basic Chat Completion

The simplest possible call. No streaming, no tools — just raw chat.


In [7]:
def chat_once(conversation: Conversation, model: str = "gpt-4o-mini") -> str:
    """
    Send a conversation to the API and return the assistant's reply.

    Exercise: implement this using client.chat.completions.create()
    Return only the text content of the first choice.
    """
    # TODO
    response = client.chat.completions.create(
        model=model,
        messages=conversation.to_api_payload(),
    )
    return response.choices[0].message.content


# Test it
conv = Conversation(system_prompt="You are a concise assistant. Reply in one sentence.")
conv.add("user", "What is the capital of France?")

reply = chat_once(conv)
print(f"Reply: {reply}")


Reply: The capital of France is Paris.


## 3. Streaming

Real agents stream — it feels alive, and it's actually better for long outputs
because you can start displaying before the full response is done.

**Key insight:** With streaming, the API returns a generator of chunks.
You accumulate them into a full string while optionally printing each chunk live.


In [8]:
def chat_stream(conversation: Conversation, model: str = "gpt-4o-mini") -> Generator[str, None, str]:
    """
    Stream the assistant's response.

    Yields: each text chunk as it arrives
    Returns: the full accumulated response (via StopIteration value)

    Usage:
        full = ""
        for chunk in chat_stream(conv):
            print(chunk, end="", flush=True)
        # full response is the generator's return value — see chat_stream_full() below
    """
    # Hint: use stream=True, then iterate response chunks
    # Each chunk: chunk.choices[0].delta.content (may be None for first/last)
    response = client.chat.completions.create(
        model=model,
        messages=conversation.to_api_payload(),
        stream=True,
    )
    for chunk in response:
        delta = chunk.choices[0].delta.content
        if delta is not None:
            yield delta


def chat_stream_full(conversation: Conversation, model: str = "gpt-4o-mini") -> str:
    """
    Stream while printing, return full text when done.
    This is the one you'll call in the agent loop.
    """
    full = ""
    for chunk in chat_stream(conversation, model):
        print(chunk, end="", flush=True)
        full += chunk
    print()  # newline after streaming ends
    return full

# Test streaming
conv = Conversation(system_prompt="You are a poet. Write 2 short haiku responses.")
conv.add("user", "Describe a running program.")
print("Streaming response:")
reply = chat_stream_full(conv)

Streaming response:
Feet pound on the ground,  
Rhythm of breath in sync sweet,  
Miles weave through the trees.  

Sunrise paints the sky,  
Determined hearts chase the dawn,  
Every step a dream.  


## 4. Token Counting & Context Management

GPT-4o has a 128k token context window. If your agent runs 24/7 (like OpenClaw),
conversation history will eventually overflow. You need a strategy.

**Three strategies:**
1. **Sliding window** — keep only last N messages
2. **Summarization** — ask the LLM to compress old history (NB-03)
3. **Vector retrieval** — store and search (full RAG, overkill here)

For now, implement token counting with `tiktoken`.


In [14]:
import tiktoken

def count_tokens(messages: list[dict], model: str = "gpt-4o-mini") -> int:
    """
    Count tokens for an OpenAI message payload.

    Rule of thumb: ~4 chars per token. But tiktoken is exact.

    Exercise: use tiktoken.encoding_for_model() to encode each message
    and count tokens. Add 3 per message for the role/formatting overhead.
    """
    # TODO
    try:
        enc = tiktoken.encoding_for_model(model)
    except:
        enc = tiktoken.get_encoding("cl100k_base")

    total = 0
    for msg in messages:
        total += 3 # role + message framing
        total += len(enc.encode(msg["content"]))
    return total


def truncate_to_fit(
    conversation: Conversation,
    max_tokens: int = 3000,
    model: str = "gpt-4o-mini"
) -> Conversation:
    """
    Remove oldest messages (not the system prompt!) until we fit in max_tokens.
    Return a new Conversation (don't mutate the original).
    """
    # TODO
    trunctated = Conversation(system_prompt=conversation.system_prompt)
    trunctated.messages = list(conversation.messages)

    while count_tokens(trunctated.to_api_payload(), model) > max_tokens:
        trunctated.messages.pop(0)
    return trunctated


In [15]:
# Test
conv = Conversation(system_prompt="You are helpful.")
for i in range(20):
    conv.add("user", f"This is message number {i} with some padding text to use tokens.")
    conv.add("assistant", f"I acknowledge message {i}. Here is a longer response to fill tokens.")

before = count_tokens(conv.to_api_payload())
after_conv = truncate_to_fit(conv, max_tokens=500)
after = count_tokens(after_conv.to_api_payload())

print(f"Before truncation: {before} tokens, {len(conv)} messages")
print(f"After truncation:  {after} tokens, {len(after_conv)} messages")
assert after <= 500
print("Token management tests pass ✓")


Before truncation: 707 tokens, 40 messages
After truncation:  497 tokens, 28 messages
Token management tests pass ✓


## 5. ChatSession — The Full Agent Core

Now wire everything together into a `ChatSession` class.
This is the object your agent will use in every interaction.


In [16]:
class ChatSession:
    """
    The core agent loop.

    Responsibilities:
    - Hold conversation history
    - Auto-truncate when approaching token limit
    - Expose .chat() for single-turn interaction
    - Expose .stream() for streaming interaction

    Design note: ChatSession knows NOTHING about tools, memory, or skills.
    Those are added in later notebooks. Clean separation of concerns.
    """

    def __init__(
        self,
        system_prompt: str,
        model: str = "gpt-4o-mini",
        max_context_tokens: int = 4000,
    ):
        self.conversation = Conversation(system_prompt=system_prompt)
        self.model = model
        self.max_context_tokens = max_context_tokens
        self._total_messages = 0

    def _prepare(self) -> Conversation:
        """Truncate history to fit context window before sending."""
        return truncate_to_fit(self.conversation, self.max_context_tokens, self.model)

    def chat(self, user_message: str) -> str:
        """Non-streaming turn."""
        self.conversation.add("user", user_message)
        prepared = self._prepare()
        reply = chat_once(prepared, self.model)
        self.conversation.add("assistant", reply)
        self._total_messages += 1
        return reply

    def stream(self, user_message: str) -> str:
        """Streaming turn — prints live, returns full reply."""
        self.conversation.add("user", user_message)
        prepared = self._prepare()
        reply = chat_stream_full(prepared, self.model)
        self.conversation.add("assistant", reply)
        self._total_messages += 1
        return reply

    def reset(self):
        """Clear history but keep system prompt."""
        self.conversation.messages.clear()

    @property
    def token_count(self) -> int:
        return count_tokens(self.conversation.to_api_payload(), self.model)

    def stats(self):
        print(f"Model:    {self.model}")
        print(f"Messages: {len(self.conversation)}")
        print(f"Tokens:   {self.token_count}")
        print(f"Total turns: {self._total_messages}")


## 6. The REPL — Talk to Your Agent

In [18]:
def run_repl(session: ChatSession):
    """
    A minimal REPL. Type 'quit' or 'exit' to stop.
    Type '/stats' to see session info.
    Type '/reset' to clear history.
    """
    print("🦞 Mini-Claw REPL started. Type 'quit' to exit.")
    print("-" * 50)
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit"):
            print("Bye!")
            break
        if user_input == "/stats":
            session.stats()
            continue
        if user_input == "/reset":
            session.reset()
            print("[History cleared]")
            continue

        print("Agent: ", end="")
        session.stream(user_input)
        print()


# Create your agent — customize the system prompt!
SYSTEM_PROMPT = """You are Claw, a sharp and capable personal AI assistant.
You are running on the user's local machine.
You are concise, thoughtful, and slightly witty.
You can help with coding, reasoning, planning, and general tasks.
When you don't know something, say so honestly."""

session = ChatSession(
    system_prompt=SYSTEM_PROMPT,
    model="gpt-4o-mini",
    max_context_tokens=4000,
)

# Uncomment to run the REPL interactively:
run_repl(session)

# For notebook testing — single turn:
reply = session.chat("Hello! What can you help me with?")
print(f"Agent: {reply}")


🦞 Mini-Claw REPL started. Type 'quit' to exit.
--------------------------------------------------
You: /stats
Model:    gpt-4o-mini
Messages: 0
Tokens:   59
Total turns: 0
You: what can you do?
Agent: I can assist you with coding, problem-solving, planning tasks, and general inquiries. Whether you need help debugging code, organizing a project, or just looking for some witty banter, I'm here for you! What do you need help with today?

You: write me a haiku
Agent: Stars twinkle above,  
Whispers of the night unfold,  
Dreams in silence soar.

You: quit
Bye!
Agent: Hello! I can help you with coding, brainstorming ideas, planning projects, answering questions, and more. If you have a specific task or question in mind, just let me know!


## 7. Exercises

**E1.** Add a `retry_on_error` wrapper around `chat_once` that retries up to 3 times with exponential backoff on `openai.RateLimitError`.

**E2.** Add a `temperature` parameter to `ChatSession` and experiment with values 0.0 (deterministic) vs 1.2 (creative). Notice how the agent's personality changes.

**E3.** Implement a `SummaryConversation` that, when token count exceeds 80% of limit, calls the LLM to summarize the oldest half of messages into a single summary message before truncating.

**E4 (hard):** Make `ChatSession` support multiple LLM backends — OpenAI and a local Ollama model — via a common interface. What's the minimal abstraction needed?

---

## ✅ Checkpoint

By now you have:
- `Conversation`: message history with token counting
- `chat_once()` / `chat_stream()`: core LLM calls
- `truncate_to_fit()`: context window management
- `ChatSession`: the full agent core

**Next:** CLAW-02 — Tool Calling. Your agent gets hands.
